# Customer Churn Prediction — corrected notebook

This notebook replaces the unrelated `tips` example and random synthetic training data with a consistent workflow using the actual churn CSV. It loads the data, inspects and cleans it, performs exploratory analysis, compares Logistic Regression and Random Forest using cross-validation on training data, evaluates the selected pipeline on a held-out test set, and saves the complete preprocessing/model pipeline.

**Before running:** If KaggleHub cannot download the dataset, set `DATA_PATH` in the configuration cell to the full path of your downloaded churn CSV. Do not report any performance score until the cells have run successfully on the real dataset.


# 1. Imports and configuration

In [ ]:
# Customer Churn Prediction — corrected end-to-end workflow
# IMPORTANT: Set DATA_PATH to your downloaded CSV path if Kaggle download is unavailable.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = None  # Example: r"C:\Users\YourName\Downloads\Telco-Customer-Churn.csv"
TARGET_CANDIDATES = ["churn", "churn label", "churn_value", "exited", "left"]
ID_HINTS = ["customerid", "customer_id", "customer id", "id"]


def find_target_column(columns):
    normalised = {str(c).strip().lower(): c for c in columns}
    for candidate in TARGET_CANDIDATES:
        if candidate in normalised:
            return normalised[candidate]
    # Fallback: target names that contain 'churn'
    for norm, original in normalised.items():
        if "churn" in norm:
            return original
    raise ValueError(
        "Could not identify the target column. Available columns: "
        + ", ".join(map(str, columns))
        + ". Set TARGET_COLUMN manually in the notebook/script."
    )


def load_dataset(data_path=None):
    """Load a local CSV, or try KaggleHub and identify a CSV containing a churn target."""
    if data_path:
        p = Path(data_path).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"DATA_PATH does not exist: {p}")
        if p.is_file():
            return pd.read_csv(p), p
        candidates = list(p.rglob("*.csv"))
    else:
        # Search common local locations first.
        candidates = []
        for folder in [Path.cwd(), Path.cwd() / "data", Path.home() / "Downloads"]:
            if folder.exists():
                candidates.extend(folder.glob("*.csv"))
        # Prefer the Kaggle dataset used in the original notebook when available.
        try:
            import kagglehub
            downloaded = Path(kagglehub.dataset_download("dhrubangtalukdar/telco-customer-churn-data"))
            candidates.extend(downloaded.rglob("*.csv"))
        except Exception as exc:
            print("KaggleHub download was not available:", exc)
            print("If necessary, download the dataset manually and set DATA_PATH to the CSV file.")

    # Choose a CSV that actually has a recognisable churn target, not an unrelated CSV.
    seen = set()
    for p in candidates:
        p = Path(p)
        if str(p) in seen or not p.is_file():
            continue
        seen.add(str(p))
        try:
            preview = pd.read_csv(p, nrows=5)
            find_target_column(preview.columns)
            return pd.read_csv(p), p
        except Exception:
            continue
    raise FileNotFoundError(
        "No CSV with a recognisable churn target was found. Download your churn CSV, "
        "then set DATA_PATH to its full path and rerun this cell."
    )


def encode_binary_target(series):
    """Convert common Yes/No, True/False, or 0/1 churn labels into 1/0."""
    s = series.copy()
    if pd.api.types.is_numeric_dtype(s):
        values = set(pd.Series(s.dropna().unique()).astype(float).tolist())
        if values.issubset({0.0, 1.0}):
            return s.astype("Int64")
    text = s.astype("string").str.strip().str.lower()
    mapping = {
        "yes": 1, "no": 0, "true": 1, "false": 0,
        "1": 1, "0": 0, "churned": 1, "stayed": 0,
        "left": 1, "retained": 0, "exited": 1
    }
    mapped = text.map(mapping)
    if mapped.notna().sum() != s.notna().sum():
        # Handle numeric strings such as "1.0" and "0.0".
        numeric = pd.to_numeric(text, errors="coerce")
        numeric_map = numeric.map({1: 1, 0: 0})
        mapped = mapped.fillna(numeric_map)
    if mapped.notna().sum() != s.notna().sum():
        bad = sorted(text[mapped.isna() & text.notna()].unique().tolist())
        raise ValueError(f"Target labels could not be mapped to 0/1. Unexpected values: {bad}")
    return mapped.astype(int)


# 2. Load the actual dataset

In [ ]:
# 1. Load the actual churn dataset. Do not replace df with demonstration/sample data.
df, dataset_file = load_dataset(DATA_PATH)
print("Dataset loaded from:", dataset_file)
print("Shape (rows, columns):", df.shape)
display(df.head())
print("\nColumn names:\n", df.columns.tolist())
df.info()


# 3. Target, data quality, and preprocessing setup

In [ ]:
# 2. Identify and prepare the target column.

target_col = find_target_column(df.columns)
print("Target column identified as:", target_col)

# Standardise column whitespace only; retain meaningful original names.
df.columns = [str(c).strip() for c in df.columns]
# Re-find target after trimming column names.
target_col = find_target_column(df.columns)

# Convert common total-charge fields from text to numeric if present.
for col in df.columns:
    if col.strip().lower().replace("_", "").replace(" ", "") in {"totalcharges", "totalcharge"}:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows where the target is missing, then encode the target.
df = df.dropna(subset=[target_col]).copy()
y = encode_binary_target(df[target_col])

# Remove target from predictors and exclude obvious record identifiers.
X = df.drop(columns=[target_col]).copy()
id_cols = [
    c for c in X.columns
    if c.strip().lower() in ID_HINTS
    or c.strip().lower().replace("_", "").replace(" ", "") == "customerid"
]
if id_cols:
    print("Dropping identifier columns:", id_cols)
    X = X.drop(columns=id_cols)

# Remove columns that contain no information.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

# 3. Basic data-quality checks.
print("\nMissing values by column (top 20):")
print(X.isna().sum().sort_values(ascending=False).head(20))
print("Duplicate full rows before de-duplication:", df.duplicated().sum())
print("Target distribution (0 = stayed, 1 = churned):")
print(y.value_counts(dropna=False).sort_index())
print("Target proportions:")
print(y.value_counts(normalize=True).sort_index().round(3))


# 4. Exploratory Data Analysis

In [ ]:
# 4. Exploratory Data Analysis using the actual dataset.
plt.figure(figsize=(5, 4))
sns.countplot(x=y.map({0: "Stayed (0)", 1: "Churned (1)"}) )
plt.title("Customer Churn Distribution")
plt.xlabel("Customer outcome")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

# Show churn rates for a few common categorical columns if they exist.
for wanted in ["Contract", "contract", "PaymentMethod", "Payment Method", "InternetService"]:
    actual = next((c for c in X.columns if c.lower() == wanted.lower()), None)
    if actual is not None and X[actual].nunique(dropna=True) <= 20:
        temp = pd.DataFrame({actual: X[actual].astype("string").fillna("Missing"), "Churn": y})
        rate = temp.groupby(actual, dropna=False)["Churn"].mean().sort_values(ascending=False)
        print(f"\nObserved churn rate by {actual} (descriptive, not causal):")
        display(rate.rename("churn_rate").to_frame())
        plt.figure(figsize=(8, 4))
        sns.barplot(x=rate.index.astype(str), y=rate.values)
        plt.title(f"Observed Churn Rate by {actual}")
        plt.ylabel("Churn rate")
        plt.xlabel(actual)
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()
        break


# 5. Train/test split and preprocessing pipeline

In [ ]:
# 5. Train/test split. The test set is held back for final evaluation.
if y.nunique() != 2:
    raise ValueError(f"Expected a binary churn target but found classes: {sorted(y.unique())}")
if y.value_counts().min() < 2:
    raise ValueError("There are fewer than 2 examples in one target class; a stratified split is not possible.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

# 6. Build preprocessing that is fitted only on training data.
numeric_features = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop")


# 6. Model comparison using training cross-validation

In [ ]:
# 7. Compare candidate models using cross-validation on training data only.
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE,
        n_jobs=-1, min_samples_leaf=2
    )
}

min_class_count = int(y_train.value_counts().min())
n_splits = min(5, min_class_count)
if n_splits < 2:
    raise ValueError("Not enough examples per class for cross-validation.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_cv_pipelines = {}

for name, classifier in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", classifier)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=None)
    results.append({"Model": name, "Mean CV F1": scores.mean(), "CV F1 Std": scores.std()})
    print(f"{name}: CV F1 = {scores.mean():.3f} +/- {scores.std():.3f}")

results_df = pd.DataFrame(results).sort_values("Mean CV F1", ascending=False)
display(results_df)
best_model_name = results_df.iloc[0]["Model"]
print("Selected model based on training cross-validation F1:", best_model_name)


# 7. Final test-set evaluation

In [ ]:
# 8. Fit the selected pipeline on all training data and evaluate once on the held-out test set.
best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[best_model_name])
])
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

print("\nFinal held-out test metrics")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1-score : {f1_score(y_test, y_pred, zero_division=0):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Stayed (0)", "Churned (1)"], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
display_labels = ["Stayed (0)", "Churned (1)"]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(values_format="d", cmap="Blues")
plt.title("Confusion Matrix — Held-out Test Set")
plt.tight_layout()
plt.show()


# 8. Feature importance

In [ ]:
# 9. Permutation importance on the original input columns.
# This estimates how much test F1 changes when each original feature is shuffled.
perm = permutation_importance(
    best_pipeline, X_test, y_test,
    scoring="f1", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top features by permutation importance (not proof of causation):")
display(importance.head(10).rename("mean_f1_drop").to_frame())
plt.figure(figsize=(8, 5))
importance.head(10).sort_values().plot(kind="barh")
plt.title("Top Features by Permutation Importance")
plt.xlabel("Mean decrease in test F1 when shuffled")
plt.tight_layout()
plt.show()


# 9. Save model and metadata

In [ ]:
# 10. Save the complete preprocessing + model pipeline, not just the classifier.
MODEL_PATH = Path("churn_prediction_pipeline.joblib")
joblib.dump(best_pipeline, MODEL_PATH)
print("Saved fitted pipeline to:", MODEL_PATH.resolve())

# Save metadata needed by a separate app or for reproducibility.
metadata = {
    "target_column": target_col,
    "target_mapping": {"stayed": 0, "churned": 1},
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "selected_model": best_model_name,
    "test_metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0))
    }
}
joblib.dump(metadata, "churn_project_metadata.joblib")

# 11. Example prediction using a real held-out customer row (input format guaranteed to match).
example_customer = X_test.iloc[[0]].copy()
example_prediction = int(best_pipeline.predict(example_customer)[0])
print("Example prediction:", "LIKELY TO CHURN" if example_prediction == 1 else "UNLIKELY TO CHURN")
if hasattr(best_pipeline, "predict_proba"):
    probability = float(best_pipeline.predict_proba(example_customer)[0][1])
    print(f"Model-estimated churn probability: {probability:.1%}")
print("\nExample input row (for inspection only):")
display(example_customer)

# NOTE: Do not hard-code a claim such as '80% accuracy'. Use the metrics printed above.
